In [0]:
import os
import json
import requests
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
catalog = dbutils.widgets.get('catalog')
bronze_schema = dbutils.widgets.get('bronze_schema')
database_id = dbutils.widgets.get('database_id')
indicator_code = dbutils.widgets.get('indicator_code')
print('Get data from Database ID: ' + database_id)
print('For Indicator Code: ' + indicator_code)
main_api_url = 'https://data360api.worldbank.org/data360/data?'
metadata_url = "https://data360api.worldbank.org/data360/metadata"
volume_path = "/Volumes/workspace/wbg_bronze/data/indicators_metadata/"
indicator_url = main_api_url + 'DATABASE_ID=' + database_id + '&INDICATOR=' + indicator_code
print('Indicator URL: ' + indicator_url)

In [0]:

def extract_indicator_data(indicator_url: str, skip: int = 0, limit: int = 1000):
    """Extracts data from the World Bank API."""

    indicator_data = []
    current_skip = skip
    
    while True:
        url = f"{indicator_url}&skip={current_skip}&top={limit}"
        response = requests.get(url)

        if response.status_code != 200:
            raise Exception("API call failed with status code: " + str(response.status_code))

        batch = response.json().get("value", [])   

        #Break if batch has no data 
        if not batch:
            break

        indicator_data.extend(batch)
        #Break if data is less than the limit 
        if len(batch) < limit:
            break

        current_skip += limit

        if not indicator_data:
            return spark.createDataFrame([], StructType([]))

        indicator_data.extend(batch)
        current_skip += limit


    columns = sorted({key for row in indicator_data for key in row.keys()})

    schema = StructType([
        StructField(col, StringType(), True)
        for col in columns
    ])

    rows_as_strings = [
        {col: None if row.get(col) is None else str(row.get(col)) for col in columns}
        for row in indicator_data
    ]

    indicator_data_df = spark.createDataFrame(rows_as_strings, schema)
    return indicator_data_df


In [0]:
indicator_data = extract_indicator_data(indicator_url)

bronze_df = (
    indicator_data
    .withColumn("database_id", lit(database_id))
    .withColumn("indicator_code", lit(indicator_code))
    .withColumn("ingestion_timestamp", current_timestamp())
)
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{indicator_code.lower()}")

In [0]:
def extract_indicator_metadata(metadata_url: str, indicator_code: str, volume_path: str):
    """Extracts metadata from the World Bank API."""

    payload = {
        "query": f"&$filter=series_description/idno eq '{indicator_code}'"
    }

    response = requests.post(metadata_url, json=payload)

    if response.status_code != 200:
        raise Exception("API metadata call failed with status code: " + str(response.status_code))

    metadata_json = response.json()
    metadata = metadata_json.get("value", [])

    if not metadata:
        raise ValueError(f"No metadata found for indicator: {indicator_code}")

    os.makedirs(volume_path, exist_ok=True)

    file_path = f"{volume_path}/{indicator_code}.json"

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(metadata_json, file, ensure_ascii=False, indent=2)

    return metadata

In [0]:
def save_indicator_metadata(metadata: list):
    if not metadata:
        raise ValueError("Metadata is empty.")

    item = metadata[0]
    series = item.get("series_description", {})

    indicator_code = series.get("idno")

    if not indicator_code:
        raise ValueError("indicator_code was not found in series_description.")

    time_periods = series.get("time_periods", [])
    first_period = time_periods[0] if time_periods else {}

    row = {
        "indicator_code": indicator_code,
        "indicator_name": series.get("name"),
        "indicator_display_name": series.get("display_name"),
        "indicator_short_definition": series.get("definition_short"),
        "indicator_long_definition": series.get("definition_long"),
        "relevance": series.get("relevance"),
        "aggregation_method": series.get("aggregation_method"),
        "measurement_unit": series.get("measurement_unit"),
        "periodicity": series.get("periodicity"),
        "database_id": series.get("database_id"),
        "database_name": series.get("database_name"),
        "start_year": first_period.get("start"),
        "end_year": first_period.get("end")
    }

    schema = StructType([
        StructField("indicator_code", StringType(), True),
        StructField("indicator_name", StringType(), True),
        StructField("indicator_display_name", StringType(), True),
        StructField("indicator_short_definition", StringType(), True),
        StructField("indicator_long_definition", StringType(), True),
        StructField("relevance", StringType(), True),
        StructField("aggregation_method", StringType(), True),
        StructField("measurement_unit", StringType(), True),
        StructField("periodicity", StringType(), True),
        StructField("database_id", StringType(), True),
        StructField("database_name", StringType(), True),
        StructField("start_year", StringType(), True),
        StructField("end_year", StringType(), True)
    ])

    metadata_df = (
        spark.createDataFrame([row], schema=schema)
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    metadata_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"indicator_code = '{indicator_code}'") \
        .partitionBy("indicator_code") \
        .saveAsTable(f"{catalog}.{bronze_schema}.indicators_metadata")

    return metadata_df

In [0]:
def save_indicator_countries(metadata: list):
    """Save indicator-country applicability data to Bronze Delta table."""

    if not metadata:
        raise ValueError("Metadata is empty.")

    item = metadata[0]
    series = item.get("series_description", {})

    indicator_code = series.get("idno")
    ref_countries = series.get("ref_country", [])

    if not indicator_code:
        raise ValueError("indicator_code was not found.")

    rows = [
        {
            "indicator_code": indicator_code,
            "country_code": country.get("code"),
            "country_name": country.get("name")
        }
        for country in ref_countries
    ]

    schema = StructType([
        StructField("indicator_code", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("country_name", StringType(), True)
    ])

    countries_df = (
        spark.createDataFrame(rows, schema=schema)
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    countries_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"indicator_code = '{indicator_code}'") \
        .partitionBy("indicator_code") \
        .saveAsTable(f"{catalog}.{bronze_schema}.indicators_countries")

    return countries_df

In [0]:
metadata = extract_indicator_metadata(metadata_url, indicator_code, volume_path)
metadata_df = save_indicator_metadata(metadata)
indicator_countries_df = save_indicator_countries(metadata)